# Tiền xử lý bước 1

## Bước này tạo file:

- df_inter.label.parquet.

File này chia dữ liệu theo x_label có 3 giá trị: 0, 1, 2 tương ứng với train, val, test.


# Train/Validation/Test data splitting

- Based on generated interactions, perform data splitting


In [1]:
import os
import pandas as pd

In [2]:
PATH = "./data/2023"

## Load interactions


In [3]:
df = pd.read_parquet(os.path.join(PATH, "df_inter.parquet"))

In [4]:
print(f"shape: {df.shape}")
df[:4]

shape: (1240219, 6)


,userID,itemID,rating,timestamp,reviewerID,asin
0,0,0,3,1657839829629,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B086QM7FVT
1,0,1,5,1655862402295,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B017IQZ9OK
2,0,2,4,1655860597696,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B08FZJ3YHH
3,0,3,5,1655860079499,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B082WJTFRR


Đoạn code này thực hiện hai thao tác xử lý dữ liệu có vẻ trái ngược nhau nhưng lại rất phổ biến trong quá trình chuẩn bị dữ liệu cho hệ thống gợi ý.


In [5]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.sort_values(by=["userID", "timestamp"], inplace=True)

df[:10]

,userID,itemID,rating,timestamp,reviewerID,asin
922815,0,4,4,1655859477958,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B004JU0H6O
513141,0,3,5,1655860079499,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B082WJTFRR
177093,0,2,4,1655860597696,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B08FZJ3YHH
333515,0,1,5,1655862402295,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B017IQZ9OK
319348,0,0,3,1657839829629,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B086QM7FVT
305315,1,51,1,1375484278000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00BE6CUWK
654098,1,50,5,1375484719000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00BEJREIC
722098,1,49,5,1394136715000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00DDMJ332
778549,1,48,3,1399996207000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00HHRH07U
685864,1,47,5,1413410307000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00MOJWOFY


In [6]:
# 1. Khai báo tên cột
uid_field, iid_field = "userID", "itemID"

# 2. Nhóm dữ liệu (Groupby)
uid_freq = df.groupby(uid_field)[iid_field]
u_i_dict = {}
for u, u_ls in uid_freq:
    u_i_dict[u] = list(u_ls)
list(u_i_dict.items())[:3]

[(0, [4, 3, 2, 1, 0]),
 (1,
  [51,
   50,
   49,
   48,
   47,
   46,
   45,
   44,
   43,
   42,
   41,
   40,
   39,
   38,
   37,
   36,
   35,
   34,
   33,
   32,
   31,
   30,
   29,
   28,
   27,
   26,
   25,
   24,
   23,
   22,
   21,
   20,
   19,
   18,
   17,
   16,
   15,
   14,
   13,
   12,
   11,
   10,
   9,
   8,
   7,
   6,
   5]),
 (2,
  [81,
   80,
   79,
   78,
   77,
   76,
   75,
   74,
   73,
   72,
   71,
   70,
   69,
   68,
   67,
   66,
   65,
   64,
   63,
   62,
   61,
   60,
   59,
   58,
   57,
   56,
   55,
   54,
   53,
   52])]

In [7]:
list(u_i_dict.keys())[:3]

[0, 1, 2]

In [8]:
new_label = []
u_ids_sorted = sorted(u_i_dict.keys())
for u in u_ids_sorted:
    items = u_i_dict[u]
    # get num interact
    n_items = len(items)
    if n_items < 10:
        # take 1 for test 1 for val and rest for train
        tmp_ls = [0] * (n_items - 2) + [1] + [2]
    else:
        # split 80% train, 10% val, 10% test
        val_test_len = int(n_items * 0.2)
        train_len = n_items - val_test_len
        val_len = val_test_len // 2
        test_len = val_test_len - val_len
        tmp_ls = [0] * train_len + [1] * val_len + [2] * test_len
    new_label.extend(tmp_ls)

new_label[:10]

[0, 0, 0, 1, 2, 0, 0, 0, 0, 0]

In [9]:
df["x_label"] = new_label
df[:20]

,userID,itemID,rating,timestamp,reviewerID,asin,x_label
922815,0,4,4,1655859477958,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B004JU0H6O,0
513141,0,3,5,1655860079499,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B082WJTFRR,0
177093,0,2,4,1655860597696,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B08FZJ3YHH,0
333515,0,1,5,1655862402295,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B017IQZ9OK,1
319348,0,0,3,1657839829629,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B086QM7FVT,2
305315,1,51,1,1375484278000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00BE6CUWK,0
654098,1,50,5,1375484719000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00BEJREIC,0
722098,1,49,5,1394136715000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00DDMJ332,0
778549,1,48,3,1399996207000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00HHRH07U,0
685864,1,47,5,1413410307000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00MOJWOFY,0


In [10]:
df.to_parquet(os.path.join(PATH, "df_inter.label.parquet"), index=False)

## Reload


In [11]:
indexed_df = pd.read_parquet(os.path.join(PATH, "df_inter.label.parquet"))
print(f"shape: {indexed_df.shape}")
indexed_df[:20]

shape: (1240219, 7)


,userID,itemID,rating,timestamp,reviewerID,asin,x_label
0,0,4,4,1655859477958,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B004JU0H6O,0
1,0,3,5,1655860079499,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B082WJTFRR,0
2,0,2,4,1655860597696,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B08FZJ3YHH,0
3,0,1,5,1655862402295,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B017IQZ9OK,1
4,0,0,3,1657839829629,AEO4M665ZOCBF7HEFRMTUDHLSB5Q,B086QM7FVT,2
5,1,51,1,1375484278000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00BE6CUWK,0
6,1,50,5,1375484719000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00BEJREIC,0
7,1,49,5,1394136715000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00DDMJ332,0
8,1,48,3,1399996207000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00HHRH07U,0
9,1,47,5,1413410307000,AFSKPY37N3C43SOI5IEXEK5JSIYA,B00MOJWOFY,0


In [12]:
train_df = indexed_df[["userID", "itemID", "rating", "timestamp", "x_label"]].copy()
train_df

,userID,itemID,rating,timestamp,x_label
0,0,4,4,1655859477958,0
1,0,3,5,1655860079499,0
2,0,2,4,1655860597696,0
3,0,1,5,1655862402295,1
4,0,0,3,1657839829629,2
...,...,...,...,...,...
1240214,150672,35568,5,1659931102331,0
1240215,150672,29043,4,1659931365259,0
1240216,150672,13132,4,1659931589085,0
1240217,150672,27903,5,1659931693722,1


In [15]:
# Lưu lại file .inter để train, tùy dataset mà tên bỏ vào thư mục data sẽ khác
# ví dụ data/baby/baby.inter, data/beauty/beauty.inter, data/clothing/clothing.inter
train_df.to_csv(os.path.join(PATH, "df_inter.label.inter"), sep="\t", index=False)

In [14]:
u_id_str, i_id_str = "userID", "itemID"
u_uni = indexed_df[u_id_str].unique()
c_uni = indexed_df[i_id_str].unique()

print(f"# of unique learners: {len(u_uni)}")
print(f"# of unique courses: {len(c_uni)}")

print("min/max of unique learners: {0}/{1}".format(min(u_uni), max(u_uni)))
print("min/max of unique courses: {0}/{1}".format(min(c_uni), max(c_uni)))

# of unique learners: 150673
# of unique courses: 35997
min/max of unique learners: 0/150672
min/max of unique courses: 0/35996
